# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema accessible at:
*https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json*

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Metadata as an object (print name and description)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Version: {getattr(dataset.metadata, 'version', None)}")
print(f"Authors: {getattr(dataset.metadata, 'author', [])}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all RecordSets with their @id
record_sets = dataset.metadata.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")

# For each RecordSet, print its fields' @id
for rs in record_sets:
    print(f"\nFields in RecordSet '{rs.name}' (@id: {rs.id}):")
    for field in rs.fields:
        print(f"  - {field.name} (@id: {field.id}) type: {getattr(field, 'data_type', None)}")

## 3. Data Extraction
Load data from one or more record sets into DataFrames using their `@id`s for downstream analysis.

In [ ]:
# Extract data from selected record sets
# Fill the variable list_of_record_set_ids and pick a record set for exploration
record_sets = dataset.metadata.record_sets

# Use all available record sets; list their @id
record_set_ids = [rs.id for rs in record_sets]

dataframes = dict()

# Load records for each record set, store in a dataframe
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    dataframes[rs.id] = pd.DataFrame(records)

# Choose a record set for demonstration: first one
main_record_set_id = record_set_ids[0] if record_set_ids else None
print(f"Columns in record set '{main_record_set_id}':")
if main_record_set_id is not None:
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing: filter records, normalize numeric fields, group by key attribute. All field accesses use `@id`s.

In [ ]:
main_rs = None
if main_record_set_id:
    # Locate the record set again to get field ids
    main_rs = [rs for rs in record_sets if rs.id == main_record_set_id][0]
    # Print all available field @id and types
    print("Available fields in main record set:")
    for f in main_rs.fields:
        print(f"  - {f.name} (@id: {f.id}) type: {getattr(f, 'data_type', None)}")

# Try to select a numeric field (prefer MW, or Peptides)
possible_numeric_ids = [f.id for f in main_rs.fields if getattr(f, 'data_type', None) in ('schema:Float','schema:Integer','Float','Integer')]
print(f"Numeric field candidate IDs: {possible_numeric_ids}")
numeric_field_id = None
for col in dataframes[main_record_set_id].columns:
    # Try common field names
    if 'MW' in col or 'weight' in col or 'Peptides' in col or 'Coverage' in col or col in possible_numeric_ids:
        numeric_field_id = col
        break
if not numeric_field_id and possible_numeric_ids:
    numeric_field_id = possible_numeric_ids[0]
print(f"Using numeric field: {numeric_field_id}")

df = dataframes[main_record_set_id]

# Make sure the numeric field is float if possible
try:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
except Exception:
    pass

threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != object else 0

filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records in '{numeric_field_id}' > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Find a possible group field: prefer fields containing 'sample', 'type', or any categorical field
group_field_id = None
for f in main_rs.fields:
    if ('sample' in f.name.lower() or 'type' in f.name.lower()) and f.id in df.columns:
        group_field_id = f.id
        break
if group_field_id is None and len(df.columns) > 1:
    # Fallback: use next non-numeric field
    for f in main_rs.fields:
        if f.id != numeric_field_id and f.id in df.columns and df[f.id].dtype == object:
            group_field_id = f.id
            break
print(f"Using group field: {group_field_id}")

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by '{group_field_id}':")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30, color='skyblue')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

# Box plot by group (if available)
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the mass spectrometry dataset using the `mlcroissant` library. We reviewed dataset structure via record sets and fields (`@id`-based), extracted records as DataFrames, performed elementary filtering, normalization, grouping, and visualized important numeric values in context.

For deeper analysis, refer to original field semantics via the record set schema and consult the dataset's domain documentation.